# Walk Forward Validation Analysis and Model Selection
In this notebook, we explore the results of the walk-forward validation experiment, portfolio performances. Eventually, these are used to selected models that will be passed on to the testing stage on completely unseen data with no look ahead bias.

### Setup:

In [37]:
# All imports
import sys
from pathlib import Path

# Get the absolute path of the directory one level up
root_dir = Path.cwd().parent

# Add it to sys.path if it's not already there
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import os
from src.utils.io import load_path_config
from src.data_processing.loading import load_csv_files
from src.evaluation.evaluator import filter_models

from src.models.registry import TradModelLibrary

In [17]:
paths_config = load_path_config(os.path.join('../config', 'paths.json'))

# Convert str paths to Path
artifacts_paths = {}
for name, path in paths_config['artifacts'].items():
    dir_path = Path('../'+ path)
    artifacts_paths[name] = dir_path

In [33]:
# Load WFV Performance files
loaded_dfs = load_csv_files(
    {
        'avg_perf': artifacts_paths['wfv_perf_dir'] / 'wfv_avg_perf_All.csv'
    }
)

avg_perf = loaded_dfs['avg_perf']
avg_perf.sort_values(by='sharpe', ascending=False, inplace=True)
print('Average Portfolio Performance Over All Steps:')
avg_perf.head(10)

Average Portfolio Performance Over All Steps:


,compunded_return,sharpe,sortino,max_drawdown,cvar,omega,calmar
S&P500,0.054988,0.114250,0.163125,-0.089762,-0.033225,1.370600,0.024025
MeanVariancePortfolio,0.096537,0.109600,0.204425,-0.103987,-0.034875,1.398138,0.019687
Equal_Weight,0.056287,0.107725,0.163887,-0.095925,-0.031587,1.374600,0.022575
BaseLSTM-custom_loss_6,0.059637,0.107438,0.165050,-0.095750,-0.031975,1.366150,0.022337
HierarchialRiskParity,0.045050,0.103025,0.157275,-0.088587,-0.027662,1.348688,0.019625
NaiveMVP,0.037525,0.078300,0.117900,-0.072063,-0.023050,1.257825,0.016362
GlobalMinimumVariance,0.037525,0.078300,0.117900,-0.072063,-0.023050,1.257825,0.016362
NestedClusteredOptimization,0.023500,0.059725,0.085875,-0.084587,-0.025412,1.221513,0.015025


## Univariate Analysis
### Overall View:

In [51]:
EQ_WT_NAME = 'Equal_Weight'
SP500_NAME = 'S&P500'

TradModelLibrary.autodiscover('src.models')
all_benches = TradModelLibrary.list_models()
all_benches.extend([EQ_WT_NAME, SP500_NAME])

# Filter models that beat Equal Weight Portfolio
filtered_perf, _ = filter_models(
    avg_perf, EQ_WT_NAME, 'sharpe', all_benches
)

filtered_perf = filtered_perf.sort_values(by='sharpe', ascending=False)
print('Models/Approaches that BEAT Equal Weight Mean Sharpe:')
filtered_perf

Models/Approaches that BEAT Equal Weight Mean Sharpe:


,compunded_return,sharpe,sortino,max_drawdown,cvar,omega,calmar
S&P500,0.054988,0.114250,0.163125,-0.089762,-0.033225,1.370600,0.024025
MeanVariancePortfolio,0.096537,0.109600,0.204425,-0.103987,-0.034875,1.398138,0.019687
Equal_Weight,0.056287,0.107725,0.163887,-0.095925,-0.031587,1.374600,0.022575
HierarchialRiskParity,0.045050,0.103025,0.157275,-0.088587,-0.027662,1.348688,0.019625
NaiveMVP,0.037525,0.078300,0.117900,-0.072063,-0.023050,1.257825,0.016362
GlobalMinimumVariance,0.037525,0.078300,0.117900,-0.072063,-0.023050,1.257825,0.016362
NestedClusteredOptimization,0.023500,0.059725,0.085875,-0.084587,-0.025412,1.221513,0.015025


In [54]:
print('Models/Approaches that DID NOT BEAT Equal Weight Mean Sharpe:')
no_beat = avg_perf.drop(filtered_perf.index, axis=0)
print(f'{len(no_beat)} NN models did not beat Equal Weight Portfolio:')
no_beat

Models/Approaches that DID NOT BEAT Equal Weight Mean Sharpe:
1 NN models did not beat Equal Weight Portfolio:


,compunded_return,sharpe,sortino,max_drawdown,cvar,omega,calmar
BaseLSTM-custom_loss_6,0.059637,0.107438,0.16505,-0.09575,-0.031975,1.36615,0.022337
